### Micro-Expression Spotting: IoU Evaluation Visualization

Loads real optical flow magnitudes from the CAS(ME)² cache and visualizes the Ground Truth, Spotted Window, Intersection, and IoU score.

> Edit the Configuration cell to change the sample or colors.

### 1. Configuration

In [ ]:
import os

ANNOTATIONS_PATH = '/home/inadio/datasets/secondaries/cas(me)^2/CAS(ME)^2code_final.xlsx'
CACHE_DIR        = '/home/inadio/datasets/secondaries/cas(me)^2/cache'

CLIP_NAME      = 'disgust2_2'
MANUAL_ONSET   = None   # override with int if needed, e.g. 396
MANUAL_OFFSET  = None   # override with int if needed, e.g. 407

CUTOFF_RATIO   = 0.30   # ApexPhaseSpotterROI cutoff_ratio for CAS(ME)^2
FPS            = 30

VIEW_MARGIN    = 35

OUTPUT_PATH    = 'iou_visual.png'
DPI            = 300

COLOR_BG_OUTER   = '#f8fafc'
COLOR_BG_INNER   = '#ffffff'
COLOR_BORDER     = '#e2e8f0'
COLOR_GRID       = '#f1f5f9'
COLOR_SIGNAL     = '#1e293b'

COLOR_SPOT_FILL  = '#ecfdf5'
COLOR_SPOT_LINE  = '#10b981'
COLOR_SPOT_LABEL = '#059669'

COLOR_GT_FILL    = '#eff6ff'
COLOR_GT_LINE    = '#3b82f6'
COLOR_GT_LABEL   = '#2563eb'

COLOR_INTER_FILL = '#ede9fe'
COLOR_INTER_LINE = '#8b5cf6'

COLOR_BADGE_TEXT = '#15803d'
COLOR_BADGE_BG   = '#f0fdf4'
COLOR_BADGE_EDGE = '#86efac'

COLOR_TITLE      = '#0f172a'
COLOR_SUBTITLE   = '#94a3b8'
COLOR_XLABEL     = '#64748b'
COLOR_XTICK      = '#94a3b8'
SUBJECT_ID     = 7   # Filter for subject s23

### 2. Imports

In [ ]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d
from IPython.display import display

from src.apex.modules.apex_phase_spotter_roi import ApexPhaseSpotterROI

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
%matplotlib inline

### 3. Load Real Magnitude Signal

In [ ]:
df_rule1 = pd.read_excel(ANNOTATIONS_PATH, sheet_name='naming rule1', header=None)
sub_map  = {int(r[2]): str(r[1]) for _, r in df_rule1.iterrows()}

df_rule2 = pd.read_excel(ANNOTATIONS_PATH, sheet_name='naming rule2', header=None)
stimulus_map = {str(r[1]): f'{int(r[0]):04d}' for _, r in df_rule2.iterrows()}

df = pd.read_excel(ANNOTATIONS_PATH, sheet_name='CASFEcode_final', header=None)
df.columns = ['Subject_ID', 'Clip_Name', 'OnsetFrame', 'ApexFrame',
               'OffsetFrame', 'AUs', 'Valence', 'Type', 'Emotion']

row = df[(df['Clip_Name'] == CLIP_NAME) & (df['Subject_ID'] == SUBJECT_ID)].iloc[0]
sub_id     = int(row['Subject_ID'])
sub_prefix = sub_map[sub_id]
stim_code  = stimulus_map[CLIP_NAME.split('_')[0]]

gt_onset   = MANUAL_ONSET  if MANUAL_ONSET  is not None else int(row['OnsetFrame'])
gt_offset  = MANUAL_OFFSET if MANUAL_OFFSET is not None else int(row['OffsetFrame'])
gt_apex    = int(row['ApexFrame'])
emotion    = str(row['Emotion'])

npz_path   = os.path.join(CACHE_DIR, f'{sub_prefix}_{stim_code}.npz')
data       = np.load(npz_path)
magnitudes = data['magnitudes'].tolist()

display(pd.DataFrame([
    {'Field': 'Sample',         'Value': f'{sub_prefix}_{CLIP_NAME}'},
    {'Field': 'Cache File',     'Value': os.path.basename(npz_path)},
    {'Field': 'Emotion',        'Value': emotion},
    {'Field': 'Total Frames',   'Value': len(magnitudes)},
    {'Field': 'GT Onset',       'Value': gt_onset},
    {'Field': 'GT Apex',        'Value': gt_apex},
    {'Field': 'GT Offset',      'Value': gt_offset},
]).set_index('Field').style.set_caption('Sample Metadata'))

,Value
Field,
Sample,s23_disgust2_2
Cache File,s23_0102.npz
Emotion,disgust
Total Frames,1070
GT Onset,526
GT Apex,530
GT Offset,540


### 4. Run Spotter & Compute IoU

In [ ]:
spotter       = ApexPhaseSpotterROI(cutoff_ratio=CUTOFF_RATIO, show_frame=False, fps=FPS)

apex_indices, phases_dict = spotter._find_apex_phase(magnitudes, phase_mode='onset_apex_offset')

def _iou(a, b):
    # Fang et al. 2023 (RMES) convention: measure = end - start (no +1).
    s_on, s_off = a; g_on, g_off = b
    inter = max(0, min(s_off, g_off) - max(s_on, g_on))
    union = (s_off - s_on) + (g_off - g_on) - inter
    return inter / union if union > 0 else 0.0

# Best-IoU match against gt (standard detection-eval assignment), not
# nearest-apex-distance selection.
if phases_dict:
    best_apex, best_phase, best_iou = None, None, -1.0
    for apex_idx, phase in phases_dict.items():
        iou_candidate = _iou((phase['start'], phase['end']), (gt_onset, gt_offset))
        if iou_candidate > best_iou:
            best_iou = iou_candidate
            best_apex, best_phase = apex_idx, phase
    spot_onset  = best_phase['start']
    spot_offset = best_phase['end']
    detected_apex = best_apex
else:
    peak_idx    = int(np.argmax(magnitudes))
    spot_onset  = max(0, peak_idx - 49)
    spot_offset = min(len(magnitudes) - 1, peak_idx + 49)
    detected_apex = peak_idx

# IoU
inter = max(0, min(spot_offset, gt_offset) - max(spot_onset, gt_onset))
union = (spot_offset - spot_onset) + (gt_offset - gt_onset) - inter
iou   = inter / union if union > 0 else 0.0
is_tp = iou >= 0.5

badge_label = f'IoU = {iou:.2f}  |  {"TRUE POSITIVE" if is_tp else "FALSE POSITIVE"}'

display(pd.DataFrame([
    {'Field': 'Spotted Onset',      'Value': spot_onset},
    {'Field': 'Detected Apex',      'Value': detected_apex},
    {'Field': 'Spotted Offset',     'Value': spot_offset},
    {'Field': 'Spotted Duration',   'Value': f'{spot_offset - spot_onset + 1} frames'},
    {'Field': 'GT Onset',           'Value': gt_onset},
    {'Field': 'GT Offset',          'Value': gt_offset},
    {'Field': 'GT Duration',        'Value': f'{gt_offset - gt_onset + 1} frames'},
    {'Field': 'Intersection',       'Value': f'{inter} frames'},
    {'Field': 'Union',              'Value': f'{union} frames'},
    {'Field': 'IoU',                'Value': f'{iou:.4f}'},
    {'Field': 'Result',             'Value': 'TRUE POSITIVE' if is_tp else 'FALSE POSITIVE'},
]).set_index('Field').style.set_caption('Spotting & IoU Results'))

### 5. Prepare Signal Window

In [5]:
event_start = min(gt_onset, spot_onset)
event_end   = max(gt_offset, spot_offset)

view_start  = max(0, event_start - VIEW_MARGIN)
view_end    = min(len(magnitudes) - 1, event_end + VIEW_MARGIN)

frames      = np.arange(view_start, view_end + 1)
raw_mag     = np.array(magnitudes)[view_start:view_end + 1]

smooth_mag  = np.array(spotter.smoothed_magnitudes)[view_start:view_end + 1]

display(pd.DataFrame([
    {'Field': 'View Start',   'Value': view_start},
    {'Field': 'View End',     'Value': view_end},
    {'Field': 'View Length',  'Value': f'{len(frames)} frames'},
]).set_index('Field').style.set_caption('Signal Crop Window'))

,Value
Field,
View Start,486
View End,575
View Length,90 frames


### 6. Plot

In [ ]:
# Normalize to [0, 1]
sig_min, sig_max = smooth_mag.min(), smooth_mag.max()
norm = lambda x: (x - sig_min) / (sig_max - sig_min + 1e-9)

sig_plot = norm(smooth_mag)
raw_plot = norm(raw_mag)

BG            = '#F4EEE5'   # Cream
SPINE         = '#D8C9B6'   # Warm Beige
GRID          = '#D8C9B6'

SIGNAL        = '#7E8B79'   # Dark Sage

GT_FILL       = '#DCE8D4'   # Light Sage
GT_LINE       = '#A8BE9B'   # Sage

TP_FILL       = '#DCE8D4'
TP_LINE       = '#7E8B79'

FP_FILL       = '#E9DDD0'
FP_LINE       = '#B59678'

OVERLAP_FILL  = '#C6D4BC'

TEXT          = '#586455'

fig, ax = plt.subplots(figsize=(10, 4.2), dpi=150)

fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

for spine in ['bottom', 'left']:
    ax.spines[spine].set_color(SPINE)
    ax.spines[spine].set_linewidth(1.0)

ax.set_yticks(np.arange(0.0, 1.01, 0.25))
ax.yaxis.set_major_formatter(ticker.NullFormatter())

ax.grid(
    True,
    linestyle=':',
    linewidth=0.8,
    color=GRID,
    alpha=0.65,
    zorder=0
)

ax.fill_betweenx(
    [0, 1.2],
    gt_onset,
    gt_offset,
    color=GT_FILL,
    alpha=0.95,
    zorder=2
)

ax.axvline(
    gt_onset,
    color=GT_LINE,
    linewidth=1.3,
    linestyle=':',
    zorder=4
)

ax.axvline(
    gt_offset,
    color=GT_LINE,
    linewidth=1.3,
    linestyle=':',
    zorder=4
)

ax.plot(
    frames,
    raw_plot,
    color=SIGNAL,
    linewidth=0.8,
    alpha=0.25,
    zorder=5
)

ax.plot(
    frames,
    sig_plot,
    color=SIGNAL,
    linewidth=2.6,
    solid_capstyle='round',
    zorder=6
)

for idx_apex, phase in phases_dict.items():

    if view_start <= idx_apex <= view_end:

        p_onset = phase['start']
        p_offset = phase['end']

        intersection = max(
            0,
            min(p_offset, gt_offset) -
            max(p_onset, gt_onset) + 1
        )

        union = (
            (p_offset-p_onset+1)
            + (gt_offset-gt_onset+1)
            - intersection
        )

        phase_iou = intersection / union if union > 0 else 0

        if phase_iou >= 0.5:
            fill_color = TP_FILL
            line_color = TP_LINE
        else:
            fill_color = FP_FILL
            line_color = FP_LINE

        ax.fill_betweenx(
            [0, 1.2],
            p_onset,
            p_offset,
            color=fill_color,
            alpha=0.85,
            zorder=1
        )

        ax.axvline(
            p_onset,
            color=line_color,
            linewidth=1.2,
            linestyle='--',
            alpha=0.85,
            zorder=4
        )

        ax.axvline(
            p_offset,
            color=line_color,
            linewidth=1.2,
            linestyle='--',
            alpha=0.85,
            zorder=4
        )

        if phase_iou > 0:

            inter_start = max(p_onset, gt_onset)
            inter_end = min(p_offset, gt_offset)

            ax.fill_betweenx(
                [0, 1.2],
                inter_start,
                inter_end,
                color=OVERLAP_FILL,
                alpha=0.9,
                zorder=3
            )

        apex_y = (
            sig_plot[frames == idx_apex][0]
            if idx_apex in frames
            else sig_plot.max()
        )

        ax.scatter(
            [idx_apex],
            [apex_y],
            s=60,
            facecolor=BG,
            edgecolor=line_color,
            linewidth=2,
            zorder=8
        )

        ax.text(
            idx_apex,
            apex_y + 0.04,
            f'Apex {idx_apex}',
            ha='center',
            va='bottom',
            fontsize=8.3,
            color=TEXT,
            fontweight='bold',
            bbox=dict(
                boxstyle='round,pad=0.25',
                facecolor=BG,
                edgecolor='none',
                alpha=0.95
            ),
            zorder=9
        )

ax.text(
    0.985,
    0.95,
    badge_label,
    transform=ax.transAxes,
    fontsize=8.5,
    fontweight='bold',
    color=TEXT,
    ha='right',
    va='top',
    bbox=dict(
        boxstyle='round,pad=0.4',
        facecolor='#E7F0E2',
        edgecolor=GT_LINE,
        linewidth=1.0
    )
)

handles = [

    mpatches.Patch(
        facecolor=TP_FILL,
        edgecolor=TP_LINE,
        linewidth=1,
        label='Spotted Window (TP)'
    ),

    mpatches.Patch(
        facecolor=FP_FILL,
        edgecolor=FP_LINE,
        linewidth=1,
        label='Spotted Window (FP)'
    ),

    mpatches.Patch(
        facecolor=GT_FILL,
        edgecolor=GT_LINE,
        linewidth=1,
        label='Ground Truth'
    ),

    mpatches.Patch(
        facecolor=OVERLAP_FILL,
        edgecolor=SIGNAL,
        linewidth=1,
        label='Overlap'
    ),

    Line2D(
        [0],
        [0],
        color=SIGNAL,
        lw=2.5,
        label='Motion Signal'
    )
]

ax.legend(
    handles=handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    frameon=False,
    fontsize=8.5,
    ncol=5,
    columnspacing=1.5,
    handlelength=1.2,
    handleheight=0.8
)

ax.set_xlabel(
    'Frame Index',
    fontsize=9,
    color=TEXT,
    labelpad=8
)

ax.set_xlim(view_start, view_end)
ax.set_ylim(0.0, 1.2)

ax.tick_params(
    axis='x',
    colors=TEXT,
    labelsize=8.5
)

ax.tick_params(
    axis='y',
    left=False
)

plt.subplots_adjust(
    bottom=0.22,
    top=0.96,
    left=0.04,
    right=0.97
)

plt.show()


## 7. Export to File

In [7]:
out_dir = os.path.dirname(OUTPUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches='tight', facecolor=fig.get_facecolor())

artifact_dir = '/home/inadio/.gemini/antigravity-cli/brain/9ac586cb-38da-407d-9b22-ea176be9ea1a'
if os.path.exists(artifact_dir):
    fig.savefig(os.path.join(artifact_dir, 'casme2_iou_visual.png'), dpi=DPI, bbox_inches='tight', facecolor=fig.get_facecolor())

display(pd.DataFrame([{
    'File': os.path.basename(OUTPUT_PATH),
    'Path': os.path.abspath(OUTPUT_PATH),
    'DPI':  DPI,
}]).style.set_caption('Exported File').hide(axis='index'))

File,Path,DPI
iou_visual.png,/home/inadio/skripkir/pulse-live/article/iou_visual.png,300
